# MD7-R2-TELL-C1 — targeted telling-boundary recalibration

This Kaggle-GPU notebook starts from frozen **MD7-R1 epoch 3** and trains three offline candidates. It corrects one narrow error: one failed meaningful scaffold remains `focus`, while persistent difficulty after multiple scaffolds may become `telling`.

Scientific constraints: ordinary four-class cross-entropy only; no weighting, RL, BKT, MRB1, LinTS, propensity adjustment, automatic epoch selection, production promotion, MathDial test, or final synthetic test. Stop after the epoch comparison table and evaluate all exported epochs with the separately frozen 48-case real-state diagnostic.


In [ ]:
# 1 — imports and exact experiment constants
import os, json, math, random, hashlib, zipfile, gc
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments, TrainerCallback

SEED = 42
MAX_LENGTH = 512
LABELS = ["generic", "probing", "focus", "telling"]
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator."
GPU_COUNT = torch.cuda.device_count()
PER_DEVICE_BATCH = 8
GRAD_ACCUM = 16 // (PER_DEVICE_BATCH * GPU_COUNT)
assert GRAD_ACCUM >= 1 and PER_DEVICE_BATCH * GPU_COUNT * GRAD_ACCUM == 16
OUTPUT_DIR = Path("/kaggle/working/md7_r2_tell_c1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("GPU count:", GPU_COUNT, "effective global batch:", PER_DEVICE_BATCH * GPU_COUNT * GRAD_ACCUM)


In [ ]:
# 2 — discover and verify the three mounted Kaggle inputs
INPUT_ROOT = Path("/kaggle/input")
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""): h.update(chunk)
    return h.hexdigest()

manifests = list(INPUT_ROOT.rglob("dataset_manifest.json"))
matches = []
for path in manifests:
    try:
        obj = json.loads(path.read_text(encoding="utf-8"))
        if obj.get("experiment") == "MD7-R2-TELL-C1": matches.append((path, obj))
    except Exception:
        pass
assert len(matches) == 1, f"Expected exactly one MD7-R2-TELL-C1 manifest, found {len(matches)}"
MANIFEST_PATH, MANIFEST = matches[0]
PACKAGE = MANIFEST_PATH.parent
CORRECTION = PACKAGE / "correction_train.jsonl"
ORIGINAL_TRAIN = PACKAGE / "telling_calibration_train_v1.jsonl"
ORIGINAL_VAL = PACKAGE / "telling_calibration_validation_v1.jsonl"
ORIGINAL_CHALLENGE = PACKAGE / "telling_boundary_challenge_val_v1.jsonl"
package_relpaths = (
    "correction_train.jsonl",
    "dataset_manifest.json",
    "dataset_audit.json",
    "telling_calibration_train_v1.jsonl",
    "telling_calibration_validation_v1.jsonl",
    "telling_boundary_challenge_val_v1.jsonl",
)
for rel in package_relpaths:
    path = PACKAGE / rel
    assert path.is_file(), path
    expected = MANIFEST["kaggle_file_hashes"].get(rel)
    if expected is not None:
        observed = sha256(path)
        assert observed == expected, (rel, observed, expected)
assert not (PACKAGE / "test.jsonl").exists(), "A protected test file was placed in the preparation dataset root."

# Resolve MathDial TRAIN and VALIDATION only. A separately mounted dataset may
# contain test.jsonl; this notebook never opens or hashes that file.
expected_md_train = MANIFEST["kaggle_file_hashes"]["mathdial/train.jsonl"]
expected_md_val = MANIFEST["kaggle_file_hashes"]["mathdial/validation.jsonl"]
local_md_train = PACKAGE / "mathdial" / "train.jsonl"
local_md_val = PACKAGE / "mathdial" / "validation.jsonl"
if (local_md_train.is_file() and local_md_val.is_file()
        and sha256(local_md_train) == expected_md_train
        and sha256(local_md_val) == expected_md_val):
    MATHDIAL_TRAIN, MATHDIAL_VAL = local_md_train, local_md_val
else:
    md_pairs = []
    for train_path in INPUT_ROOT.rglob("train.jsonl"):
        val_path = train_path.parent / "validation.jsonl"
        if not val_path.is_file():
            continue
        lower = str(train_path.parent).lower()
        score = (100 if "mathdial" in lower else 0) + (10 if "move-selection" in lower else 0)
        md_pairs.append((score, train_path, val_path))
    md_pairs.sort(key=lambda x: (-x[0], str(x[1])))
    selected_md = None
    for _, train_path, val_path in md_pairs:
        if sha256(train_path) == expected_md_train and sha256(val_path) == expected_md_val:
            selected_md = (train_path, val_path)
            break
    assert selected_md is not None, (
        "Could not find the exact authorized MathDial train.jsonl and validation.jsonl under /kaggle/input. "
        "Attach the mathdial-move-selection dataset."
    )
    MATHDIAL_TRAIN, MATHDIAL_VAL = selected_md

# Resolve the complete frozen MD7-R1 Hugging Face export. Notebook-output
# mounts are supported; loose epoch state_dict files are intentionally rejected.
expected_model = MANIFEST["kaggle_file_hashes"]["md7r1_epoch3/model.safetensors"]
required_model_hashes = (
    ("config.json", MANIFEST["kaggle_file_hashes"]["md7r1_epoch3/config.json"]),
    ("tokenizer.json", MANIFEST["kaggle_file_hashes"]["md7r1_epoch3/tokenizer.json"]),
    ("tokenizer_config.json", MANIFEST["kaggle_file_hashes"]["md7r1_epoch3/tokenizer_config.json"]),
)

def model_dir_matches(path):
    weight = path / "model.safetensors"
    if not weight.is_file() or sha256(weight) != expected_model:
        return False
    for name, expected in required_model_hashes:
        file_path = path / name
        if not file_path.is_file() or sha256(file_path) != expected:
            return False
    try:
        config = json.loads((path / "config.json").read_text(encoding="utf-8"))
        labels = [config["id2label"][str(i)] for i in range(4)]
    except Exception:
        return False
    return labels == ID2LABEL

local_model = PACKAGE / "md7r1_epoch3"
if local_model.is_dir() and model_dir_matches(local_model):
    MODEL_DIR = local_model
else:
    model_candidates = []
    for config_path in INPUT_ROOT.rglob("config.json"):
        model_path = config_path.parent
        if not (model_path / "model.safetensors").is_file():
            continue
        lower = str(model_path).lower()
        score = 0
        score += 200 if "md7r1_epoch3" in lower else 0
        score += 80 if "md7r1" in lower else 0
        score += 40 if "epoch3" in lower else 0
        score += 20 if "results" in lower else 0
        score -= 200 if any(x in lower for x in ("md6", "mrb", "telling_calibration", "epoch1", "epoch2")) else 0
        model_candidates.append((score, model_path))
    model_candidates.sort(key=lambda x: (-x[0], str(x[1])))
    MODEL_DIR = None
    for _, model_path in model_candidates:
        if model_dir_matches(model_path):
            MODEL_DIR = model_path
            break
    assert MODEL_DIR is not None, (
        "Could not find the exact complete MD7-R1 epoch3 model export under /kaggle/input. "
        "Attach the MD7-R1 AdaptMath notebook output containing config/tokenizer files and model.safetensors."
    )

assert sha256(CORRECTION) == "7ecc68d03022d19ee755b4e714c1258ce5808bd3a13bf55a0c8a0d5a8f86aa90"
assert MANIFEST["guardrails"]["mathdial_final_test_use"] == 0
assert MANIFEST["guardrails"]["mrbench_v3_test_use"] == 0
assert MANIFEST["guardrails"]["production_changes"] == 0
print("Preparation package:", PACKAGE)
print("MathDial train:", MATHDIAL_TRAIN)
print("MathDial validation:", MATHDIAL_VAL)
print("Frozen MD7-R1 model:", MODEL_DIR)
print("INPUT HASH GUARD: PASS")


In [ ]:
# 3 — load only authorized training/validation inputs
def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

correction = read_jsonl(CORRECTION)
original_train = read_jsonl(ORIGINAL_TRAIN)
original_val = read_jsonl(ORIGINAL_VAL)
challenge_val = read_jsonl(ORIGINAL_CHALLENGE)
mathdial_train = read_jsonl(MATHDIAL_TRAIN)
mathdial_val = read_jsonl(MATHDIAL_VAL)

assert len(correction) == 2400 and Counter(r["target_move"] for r in correction) == {"focus": 1200, "telling": 1200}
assert len(original_train) == 6000 and Counter(r["target_move"] for r in original_train) == {x: 1500 for x in LABELS}
assert len(original_val) == 2400 and Counter(r["target_move"] for r in original_val) == {x: 600 for x in LABELS}
assert len(challenge_val) == 800
assert len(mathdial_train) == 14905 and len(mathdial_val) == 1850
print("AUTHORIZED INPUT COUNTS: PASS")


In [ ]:
# 4 — exact 70/30 mixture using MathDial TRAIN only
def norm_synthetic(rows, source):
    return [{"example_id": str(r["example_id"]), "problem": r["problem"], "history": r["history"],
             "target_move": r["target_move"], "source": source, "history_format": "preformatted_string"} for r in rows]
def norm_mathdial(rows, source):
    return [{"example_id": str(r["example_id"]), "problem": r["problem"], "history": r["history"],
             "target_move": r["target_move"], "source": source, "history_format": "mathdial_turn_list"} for r in rows]
def proportional_stratified_sample(rows, total_n, seed):
    rng = random.Random(seed)
    by_label = {label: [r for r in rows if r["target_move"] == label] for label in LABELS}
    counts = {label: len(by_label[label]) for label in LABELS}; total = sum(counts.values())
    raw = {label: total_n * counts[label] / total for label in LABELS}
    quota = {label: math.floor(raw[label]) for label in LABELS}
    order = sorted(LABELS, key=lambda label: (raw[label]-quota[label], label), reverse=True)
    for label in order[:total_n-sum(quota.values())]: quota[label] += 1
    sample = []
    for label in LABELS: sample.extend(rng.sample(by_label[label], quota[label]))
    rng.shuffle(sample); return sample, quota

original_rows = norm_synthetic(original_train, "original_telling_calibration_train")
correction_rows = norm_synthetic(correction, "md7_r2_tell_c1_correction_train")
md_train_rows = norm_mathdial(mathdial_train, "mathdial_train_replay")
md_val_rows = norm_mathdial(mathdial_val, "mathdial_validation")
md_replay, replay_quota = proportional_stratified_sample(md_train_rows, 3600, SEED)
train_rows = original_rows + correction_rows + md_replay
random.Random(SEED).shuffle(train_rows)
assert len(train_rows) == 12000 and replay_quota == {"generic":864,"probing":824,"focus":1315,"telling":597}
assert Counter(r["source"] for r in train_rows) == {"original_telling_calibration_train":6000,"md7_r2_tell_c1_correction_train":2400,"mathdial_train_replay":3600}
print("TRAINING MIXTURE:", Counter(r["source"] for r in train_rows), Counter(r["target_move"] for r in train_rows), replay_quota)


In [ ]:
# 5 — exact deployed MD7-R1 paired preprocessing
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True, use_fast=True)
tokenizer.truncation_side = "left"
def format_mathdial_history(history):
    if not history: return "[No previous conversation]"
    assert isinstance(history, list)
    return "\n".join(f"{turn['user']}: {turn['text']}" for turn in history)
def normalized_history(row):
    if row["history_format"] == "mathdial_turn_list": return format_mathdial_history(row["history"])
    text = str(row["history"]); return text if text.strip() else "[No previous conversation]"
def paired_text(row):
    return "Problem:\n" + str(row["problem"]), "Conversation:\n" + normalized_history(row) + "\n\nNext teacher pedagogical move:"
def encode(row):
    first, second = paired_text(row)
    return tokenizer(first, second, truncation="only_second", max_length=MAX_LENGTH)
class MoveDataset(Dataset):
    def __init__(self, rows): self.rows = list(rows)
    def __len__(self): return len(self.rows)
    def __getitem__(self, index):
        row = self.rows[index]; item = encode(row); item["labels"] = LABEL2ID[row["target_move"]]; return item
collator = DataCollatorWithPadding(tokenizer=tokenizer)
train_ds = MoveDataset(train_rows)
target_val_ds = MoveDataset(norm_synthetic(original_val, "original_telling_calibration_validation"))
challenge_val_ds = MoveDataset(norm_synthetic(challenge_val, "telling_boundary_challenge_validation"))
md_val_ds = MoveDataset(md_val_rows)
print(paired_text(md_val_rows[0])[0]); print(paired_text(md_val_rows[0])[1][:1000])
assert tokenizer.truncation_side == "left"


In [ ]:
# 6 — evaluation helpers
def metrics_from_logits(logits, labels):
    pred = np.argmax(logits, axis=-1)
    per = f1_score(labels, pred, average=None, labels=[0,1,2,3], zero_division=0)
    return {"accuracy":float(accuracy_score(labels,pred)), "macro_f1":float(f1_score(labels,pred,average="macro",zero_division=0)),
            "per_class_f1":{LABELS[i]:float(per[i]) for i in range(4)},
            "pred_distribution":{LABELS[i]:int((pred==i).sum()) for i in range(4)}}
def challenge_metrics(logits, labels):
    out = metrics_from_logits(logits, labels); pred=np.argmax(logits,axis=-1); y=np.asarray(labels); tell=3
    tp=int(((pred==tell)&(y==tell)).sum()); fp=int(((pred==tell)&(y!=tell)).sum()); fn=int(((pred!=tell)&(y==tell)).sum()); non=int((y!=tell).sum())
    out.update({"telling_precision":tp/(tp+fp) if tp+fp else 0.0,"telling_recall":tp/(tp+fn) if tp+fn else 0.0,
                "telling_false_trigger_rate_on_non_telling":fp/non if non else 0.0,"telling_tp":tp,"telling_fp":fp,"telling_fn":fn})
    return out
@torch.no_grad()
def predict_dataset(model, dataset, batch_size=16):
    model.eval(); loader=torch.utils.data.DataLoader(dataset,batch_size=batch_size,shuffle=False,collate_fn=collator)
    logits=[]; labels=[]; device=next(model.parameters()).device
    for batch in loader:
        labels.append(batch.pop("labels").numpy()); batch={k:v.to(device) for k,v in batch.items()}; logits.append(model(**batch).logits.detach().cpu().numpy())
    return np.concatenate(logits), np.concatenate(labels)


In [ ]:
# 7 — REQUIRED PRETRAINING GUARD: frozen baseline on MathDial VALIDATION
config = json.loads((MODEL_DIR/"config.json").read_text(encoding="utf-8"))
assert {int(k):v for k,v in config["id2label"].items()} == ID2LABEL and config["label2id"] == LABEL2ID
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR, local_files_only=True).to("cuda")
md_logits, md_labels = predict_dataset(model, md_val_ds)
baseline_md = metrics_from_logits(md_logits, md_labels)
expected = {"accuracy":0.5162162162162162,"macro_f1":0.4611286390258053,
            "per_class_f1":{"generic":0.6105263157894737,"probing":0.3333333333333333,"focus":0.6004672897196262,"telling":0.300187617260788},
            "pred_distribution":{"generic":413,"probing":188,"focus":1034,"telling":215}}
assert abs(baseline_md["accuracy"]-expected["accuracy"]) <= 1e-12
assert abs(baseline_md["macro_f1"]-expected["macro_f1"]) <= 1e-12
assert all(abs(baseline_md["per_class_f1"][k]-v) <= 1e-12 for k,v in expected["per_class_f1"].items())
assert baseline_md["pred_distribution"] == expected["pred_distribution"]
(OUTPUT_DIR/"baseline_mathdial_validation.json").write_text(json.dumps(baseline_md,indent=2))
print("PRETRAINING GUARD: PASS", json.dumps(baseline_md,indent=2))
del model, md_logits; gc.collect(); torch.cuda.empty_cache()


In [ ]:
# 8 — export and evaluate every epoch; no automatic selection
EPOCH_RESULTS=[]
class EpochAuditCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        epoch=int(round(state.epoch)); model_=kwargs["model"]; path=OUTPUT_DIR/f"epoch{epoch}"; path.mkdir(parents=True,exist_ok=True)
        model_.save_pretrained(path); tokenizer.save_pretrained(path)
        row={"epoch":epoch}
        for name,ds,fn in [("original_synthetic_val",target_val_ds,metrics_from_logits),("telling_challenge_val",challenge_val_ds,challenge_metrics),("mathdial_val",md_val_ds,metrics_from_logits)]:
            logits,labels=predict_dataset(model_,ds); row[name]=fn(logits,labels)
        EPOCH_RESULTS.append(row); (OUTPUT_DIR/f"epoch{epoch}_metrics.json").write_text(json.dumps(row,indent=2)); print(json.dumps(row,indent=2))
        return control


In [ ]:
# 9 — TRAIN ON KAGGLE GPU ONLY: ordinary four-class cross-entropy
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR, local_files_only=True, num_labels=4, id2label=ID2LABEL, label2id=LABEL2ID)
args=TrainingArguments(output_dir=str(OUTPUT_DIR/"trainer_tmp"),learning_rate=1e-5,num_train_epochs=3,
    per_device_train_batch_size=PER_DEVICE_BATCH,per_device_eval_batch_size=16,gradient_accumulation_steps=GRAD_ACCUM,
    weight_decay=0.01,warmup_ratio=0.10,logging_strategy="epoch",save_strategy="no",load_best_model_at_end=False,
    fp16=True,report_to="none",seed=SEED,data_seed=SEED)
trainer=Trainer(model=model,args=args,train_dataset=train_ds,data_collator=collator,callbacks=[EpochAuditCallback()])
print("INITIALIZATION:", MODEL_DIR); print("ROWS:",len(train_rows),"LOSS: ordinary cross-entropy")
trainer.train()


In [ ]:
# 10 — validation-only epoch table; DO NOT select or promote here
rows=[]
for item in EPOCH_RESULTS:
    rows.append({"epoch":item["epoch"],"mathdial_accuracy":item["mathdial_val"]["accuracy"],"mathdial_macro_f1":item["mathdial_val"]["macro_f1"],
      **{f"mathdial_f1_{k}":v for k,v in item["mathdial_val"]["per_class_f1"].items()},
      **{f"mathdial_pred_{k}":v for k,v in item["mathdial_val"]["pred_distribution"].items()},
      "synthetic_val_macro_f1":item["original_synthetic_val"]["macro_f1"],"challenge_telling_precision":item["telling_challenge_val"]["telling_precision"],
      "challenge_telling_recall":item["telling_challenge_val"]["telling_recall"],"challenge_false_telling_rate":item["telling_challenge_val"]["telling_false_trigger_rate_on_non_telling"]})
comparison=pd.DataFrame(rows); display(comparison); comparison.to_csv(OUTPUT_DIR/"epoch_comparison.csv",index=False)
assert sorted(comparison.epoch.tolist()) == [1,2,3]
print("STOP: export all epochs. Do not select automatically and do not evaluate a final synthetic test.")


In [ ]:
# 11 — package all three offline candidates
zip_path=Path("/kaggle/working/md7_r2_tell_c1_epoch_candidates.zip")
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as archive:
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file() and "trainer_tmp" not in str(path): archive.write(path,path.relative_to(OUTPUT_DIR.parent))
print("Download:",zip_path)
